## Introduction

This notebook evaluates the Python code predictions generated by the LLM for **Setting A** using an automated smoke testing pipeline. Each script is first validated for syntax correctness using `py_compile`, and then executed inside the project virtual environment to detect runtime errors. To ensure feasible execution on limited computational resources, the scripts are run under a **FAST_EVAL** configuration (e.g., reduced epochs, smaller datasets, and disabled blocking visualizations). For every sample, the notebook records pass/fail status, execution time, and relevant output logs. Network-related dataset issues are reported separately from genuine code-level failures.


### Locate prediction scripts and set runtime directory

This cell defines the root folder containing all `prediction.py` files, creates a dedicated runtime directory for evaluation outputs, and collects the list of prediction scripts to be tested.


In [ ]:
import os, json, time, subprocess
from pathlib import Path

ROOT = Path(r"C:\Users\hbahmanyar\MentorApp\eval_outputs")
RUNTIME = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast")
RUNTIME.mkdir(parents=True, exist_ok=True)

pred_files = sorted(ROOT.rglob("prediction.py"))
len(pred_files)


88

### Syntax validation with `py_compile`

This cell performs a fast syntax-only check for every `prediction.py` file using `py_compile`. It records whether each script compiles successfully, along with the elapsed time and any syntax error message. The full results are saved to `syntax_report.json`, and the cell prints a summary of how many scripts passed vs. failed.


In [30]:
import py_compile

syntax_results = []
for i, f in enumerate(pred_files, 1):
    t0 = time.time()
    try:
        py_compile.compile(str(f), doraise=True)
        ok = True
        err = ""
    except Exception as e:
        ok = False
        err = repr(e)

    syntax_results.append({
        "idx": i,
        "file": str(f),
        "ok": ok,
        "seconds": round(time.time() - t0, 4),
        "error": err
    })

syntax_out = RUNTIME / "syntax_report.json"
syntax_out.write_text(json.dumps(syntax_results, indent=2), encoding="utf-8")

sum(r["ok"] for r in syntax_results), len(syntax_results) - sum(r["ok"] for r in syntax_results)


(88, 0)

**CONCLUSION:**

✅ All **88** scripts passed the `py_compile` syntax check (**0** syntax failures).


### Create FAST_EVAL patched versions of each script

This cell defines a lightweight patching step that rewrites the original `prediction.py` files into a `patched/` runtime folder. The patch enables a `FAST_EVAL` mode to keep execution feasible by forcing `epochs=1`, limiting training steps, skipping blocking plots (`plt.show()`), and shrinking CIFAR-10 data when detected. The helper `materialize_patched()` generates and saves the patched script while preserving the original folder structure.


In [4]:
import re

WORK = RUNTIME / "patched"
WORK.mkdir(parents=True, exist_ok=True)

TIMEOUT = 90
FORCE_CPU = False  # set True if you want to avoid GPU usage

def patch_fast_eval(code: str) -> str:
    header = r'''
import os
FAST_EVAL = os.environ.get("FAST_EVAL", "0") == "1"
if FAST_EVAL:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
'''
    code = header + "\n" + code

    # cap epochs
    code = re.sub(r"(epochs\s*=\s*)\d+", r"\g<1>1", code)

    # cap steps
    code = re.sub(r"(steps_per_epoch\s*=\s*)\d+", r"\g<1>1", code)
    code = re.sub(r"(validation_steps\s*=\s*)\d+", r"\g<1>1", code)

    # disable plt.show
    code = re.sub(r"\bplt\.show\(\)", "print('[FAST_EVAL] plt.show() skipped')", code)

    # shrink CIFAR pattern if present
    shrink = r'''
if FAST_EVAL:
    try:
        x_train = x_train[:512]; y_train = y_train[:512]
        x_test  = x_test[:128]; y_test  = y_test[:128]
    except Exception:
        pass
'''
    code = re.sub(
        r"(=\s*cifar10\.load_data\(\)\s*)",
        r"\1\n" + shrink + "\n",
        code
    )

    return code

def materialize_patched(src: Path) -> Path:
    rel = src.relative_to(ROOT)
    dst = WORK / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    code = src.read_text(encoding="utf-8", errors="ignore")
    dst.write_text(patch_fast_eval(code), encoding="utf-8")
    return dst


### Execute a patched script under FAST_EVAL and capture logs

This helper function runs a single patched `prediction.py` file in a subprocess with `FAST_EVAL=1` enabled (and optional CPU-only mode). It captures the return code, runtime, and the tail of both stdout and stderr for debugging. If execution_


In [5]:
import sys

def run_script(path: Path, timeout=TIMEOUT):
    env = os.environ.copy()
    env["FAST_EVAL"] = "1"
    if FORCE_CPU:
        env["CUDA_VISIBLE_DEVICES"] = ""

    t0 = time.time()
    try:
        proc = subprocess.run(
            [sys.executable, str(path)],
            cwd=str(path.parent),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout
        )
        dt = time.time() - t0
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-30:]),
            "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-60:]),
        }
    except subprocess.TimeoutExpired as e:
        dt = time.time() - t0
        out = e.stdout or ""
        err = e.stderr or ""
        return {
            "ok": False,
            "returncode": None,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join(out.splitlines()[-30:]),
            "stderr_tail": "TIMEOUT\n" + "\n".join(err.splitlines()[-60:]),
        }


### Run the full smoke test and save `smoke_report.json`

This cell executes the complete smoke evaluation over all prediction scripts. Scripts that failed the syntax check are skipped, while valid scripts are first patched into FAST_EVAL mode and then executed with a timeout. The results (pass/fail, runtime, and log tails) are saved to `smoke_report.json`, and the cell prints a summary of passed, failed, and skipped samples.


In [ ]:
syntax_ok = {r["file"] for r in syntax_results if r["ok"]}

smoke_results = []
for i, src in enumerate(pred_files, 1):
    if str(src) not in syntax_ok:
        smoke_results.append({
            "idx": i,
            "file": str(src),
            "ok": False,
            "skipped": True,
            "reason": "syntax_failed"
        })
        continue

    patched = materialize_patched(src)
    r = run_script(patched)
    r.update({
        "idx": i,
        "file": str(src),
        "patched": str(patched),
        "skipped": False
    })
    smoke_results.append(r)

smoke_out = RUNTIME / "smoke_report.json"
smoke_out.write_text(json.dumps(smoke_results, indent=2), encoding="utf-8")

passed = sum(r.get("ok", False) for r in smoke_results if not r.get("skipped", False))
failed = sum((not r.get("ok", False)) for r in smoke_results if not r.get("skipped", False))
skipped = sum(r.get("skipped", False) for r in smoke_results)
passed, failed, skipped


(60,
 28,
 0,
 WindowsPath('C:/Users/hbahmanyar/MentorApp/eval_runtime_fast/smoke_report.json'))

### Inspect a sample of failed runs

This cell filters the smoke test results to only the scripts that failed at runtime (excluding skipped files) and prints a readable summary for the first few failures. For each failed sample, it shows the index, file path, execution time, return code, and the tail of the stderr log to quickly identify the cause of failure.


In [ ]:
fails = [r for r in smoke_results if (not r.get("skipped", False)) and (not r.get("ok", False))]
for r in fails[:10]:
    print("="*100)
    print(f'[{r["idx"]}] {r["file"]}  ({r["seconds"]}s)  returncode={r["returncode"]}')
    print("--- stderr tail ---")
    print(r["stderr_tail"])


**Note:** Some failures were caused by timeout limits and external dataset access issues (e.g., SSL/HTTP errors). In the next step, we increased the timeout threshold and replaced the affected online datasets with offline alternatives, then reran those 8 samples under the updated conditions.


### Categorize runtime failures by error type

This cell loads the saved `smoke_report.json`, filters out the failed (non-skipped) runs, and classifies each failure based on patterns in the captured stderr output. The final `Counter` summary provides a quick breakdown of how many failures are due to timeouts, SSL/network issues, encoding/import environment problems, or other runtime errors.


In [ ]:
import json, re
from pathlib import Path

smoke_path = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast\smoke_report.json")
smoke = json.loads(smoke_path.read_text(encoding="utf-8"))

fails = [r for r in smoke if (not r.get("skipped", False)) and (not r.get("ok", False))]

def classify(stderr_tail: str):
    s = (stderr_tail or "")
    if "TIMEOUT" in s:
        return "TIMEOUT"
    if "SSLCertVerificationError" in s or "CERTIFICATE_VERIFY_FAILED" in s:
        return "NETWORK_SSL"
    if "UnicodeDecodeError" in s and "charmap" in s:
        return "ENV_ENCODING"
    if "ModuleNotFoundError" in s or "ImportError" in s:
        return "ENV_IMPORT"
    return "OTHER_RUNTIME"

from collections import Counter
Counter(classify(r.get("stderr_tail","")) for r in fails)


Counter({'OTHER_RUNTIME': 16, 'TIMEOUT': 4, 'NETWORK_SSL': 4, 'ENV_IMPORT': 4})

### Rerun selected failures with updated runtime conditions

This cell defines a second execution helper that reruns scripts under more robust conditions. It increases the timeout limit, enforces UTF-8 output decoding to avoid Windows encoding issues, optionally suppresses verbose TensorFlow logs, and applies certificate settings (via `certifi`) to improve HTTPS dataset downloads. The function returns the same structured pass/fail result with execution time and log tails, making it compatible with the original smoke report format.


In [10]:
import os, sys, time, subprocess

NEW_TIMEOUT = 300     # increase as you like (seconds)
FORCE_CPU = False     # keep as you want

def run_script_new_conditions(path: Path, timeout=NEW_TIMEOUT):
    env = os.environ.copy()
    env["FAST_EVAL"] = "1"

    # Make output decoding stable
    env["PYTHONUTF8"] = "1"

    # Optional: silence TF info spam
    env["TF_CPP_MIN_LOG_LEVEL"] = "2"

    if FORCE_CPU:
        env["CUDA_VISIBLE_DEVICES"] = ""

    # SSL environment help (certifi)
    try:
        import certifi
        env["SSL_CERT_FILE"] = certifi.where()
        env["REQUESTS_CA_BUNDLE"] = certifi.where()
    except Exception:
        pass

    t0 = time.time()
    try:
        proc = subprocess.run(
            [sys.executable, str(path)],
            cwd=str(path.parent),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout
        )
        dt = time.time() - t0
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-30:]),
            "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-80:]),
        }
    except subprocess.TimeoutExpired as e:
        dt = time.time() - t0
        out = e.stdout or ""
        err = e.stderr or ""
        return {
            "ok": False,
            "returncode": None,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join(str(out).splitlines()[-30:]),
            "stderr_tail": "TIMEOUT\n" + "\n".join(str(err).splitlines()[-80:]),
        }


### Rerun TIMEOUT and network-related failures

This cell selects the failed samples that were classified as **TIMEOUT** or **NETWORK_SSL** and reruns only those cases using the updated execution settings (higher timeout + im**_**


In [ ]:
from pathlib import Path

patched_root = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast\patched")

to_rerun_categories = {"NETWORK_SSL", "TIMEOUT"}  

rerun_results = []
for r in fails:
    cat = classify(r.get("stderr_tail",""))
    if cat not in to_rerun_categories:
        continue

    # You ran patched scripts before; rerun the patched version if available
    patched = r.get("patched")
    if patched:
        script_path = Path(patched)
    else:
        # fallback: map original file -> patched location
        src = Path(r["file"])
        rel = src.relative_to(Path(r"C:\Users\hbahmanyar\MentorApp\eval_outputs"))
        script_path = patched_root / rel

    out = run_script_new_conditions(script_path, timeout=NEW_TIMEOUT)
    out.update({
        "idx": r["idx"],
        "file": r["file"],
        "patched": str(script_path),
        "previous_category": cat,
        "previous_seconds": r.get("seconds"),
    })
    rerun_results.append(out)

len(rerun_results)


4

In [31]:
import json
from collections import Counter
from pathlib import Path

out_path = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast\smoke_rerun_report2.json")
out_path.write_text(json.dumps(rerun_results, indent=2), encoding="utf-8")

Counter("pass" if x["ok"] else "fail" for x in rerun_results)


Counter({'pass': 4})

**Note:** First, I resolved the 4 timeout failures by increasing the execution limit from **90s** to **300s**. Then, I addressed the remaining 4 network-related failures by replacing the online dataset download with a similar **offline** dataset, allowing those scripts to run successfully without external access.


### Compute final smoke-test pass rate

This cell loads the saved smoke report and computes the final evaluation statistics. It counts how many samples **passed**, **failed**, and were **skipped**, then reports the pass percentage both over the full dataset and over evaluated samples only (excluding skipped entries). This provides the main runtime correctness metric used in the notebook summary.


In [32]:
import json
from pathlib import Path

REPORT_PATH = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast\smoke_report.json")
# If you merged reruns, use:
# REPORT_PATH = Path(r"C:\Users\hbahmanyar\MentorApp\eval_runtime_fast\smoke_report_merged.json")

rows = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

total = len(rows)
passed = sum(1 for r in rows if r.get("ok") is True and not r.get("skipped", False))
failed = sum(1 for r in rows if r.get("ok") is False and not r.get("skipped", False))
skipped = sum(1 for r in rows if r.get("skipped", False))

# Percentages
pass_pct_total = (passed / total * 100) if total else 0.0
evaled = passed + failed
pass_pct_evaled = (passed / evaled * 100) if evaled else 0.0


print(f"Total samples: {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Skipped:       {skipped}")
print()
print(f"Pass % (of total):   {pass_pct_total:.2f}%")
print(f"Pass % (of evaluated, excluding skipped): {pass_pct_evaled:.2f}%")


Total samples: 88
Passed:        68
Failed:        20
Skipped:       0

Pass % (of total):   77.27%
Pass % (of evaluated, excluding skipped): 77.27%


## Conclusion

The final smoke evaluation was conducted on a total of **88** generated samples. After resolving timeout issues (by increasing the execution limit) and replacing blocked online datasets with suitable offline alternatives, the majority of scripts executed successfully under the FAST_EVAL configuration.

One sample initially failed due to an assertion error caused by reducing the training epochs from 50 to 1 during FAST_EVAL. After manually restoring the original 50 epochs, the script executed correctly, confirming that the failure was not due to a code bug but to the evaluation constraint.

Considering this correction, the final results are:

- **Passed:** 69  
- **Failed:** 19  
- **Total:** 88  
- **Final Pass Rate:** **78.40%**

These results indicate that nearly four out of five generated programs are runtime-correct under controlled evaluation settings, establishing a strong baseline for further refinement.


**Next Step:** In the following phase, I will prompt the model again using the previously generated predictions along with their corresponding runtime errors, asking it to correct those specific issues. This will allow us to measure the final acceptance accuracy after error-aware refinement. So far, a **78% pass rate** provides a reasonable and promising baseline performance for the model.
